# Week 47, Day 5: Sleep and the slow oscillation: up and down states

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w047/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Noise-driven switching against a slow oscillation

Complete run(): Euler steps of both equations with dt = 2 ms, using pre-drawn noise. Then compare b = 0, I = 0 with b = 2, I = 1 over 30 s using updown(), which returns the fraction of time up, mean up and down durations and transitions per second.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
f = lambda u: 1 / (1 + np.exp(-(u - 3) / 0.5))

def run(I=0.0, b=1.0, w=6.0, tau=0.01, taua=0.5, T=30.0, dt=0.002, noise=0.15, seed=0):
    g = np.random.default_rng(seed); n = int(T / dt)
    x, a = 0.0, 0.0; X = np.zeros(n); A = np.zeros(n)
    nz = noise * np.sqrt(dt / tau) * g.normal(size=n)
    for k in range(n):
        ...
        X[k], A[k] = x, a
    return X, A

def updown(X, dt=0.002):
    up = X > 0.5; ch = np.diff(up.astype(int)); on = np.where(ch == 1)[0]; off = np.where(ch == -1)[0]
    rate = len(on) / (len(X) * dt)
    if len(on) < 3 or len(off) < 3:
        return up.mean(), np.nan, np.nan, rate
    if off[0] < on[0]:
        off = off[1:]
    m = min(len(on), len(off))
    return up.mean(), ((off[:m] - on[:m]) * dt).mean(), ((on[1:m] - off[: m - 1]) * dt).mean(), rate

runs = {"no adaptation": run(I=0.0, b=0.0), "adaptation": run(I=1.0, b=2.0)}
stats = {k: updown(v[0]) for k, v in runs.items()}
for k, s in stats.items():
    print(f"{k:13s}: up {100 * s[0]:.0f}% of the time, mean up {s[1]:.2f} s, mean down {s[2]:.2f} s, {s[3]:.2f} transitions per second")
fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
for a_, (k, (X, A)) in zip(ax, runs.items()):
    t = np.arange(len(X)) * 0.002; a_.plot(t, X); a_.plot(t, A / 2, alpha=0.6); a_.set_title(k)
ax[-1].set_xlabel("time (s)"); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_noise_only():
    "without adaptation, transitions are rare (fewer than 0.3 per second)"
    assert stats["no adaptation"][3] < 0.3
def test_oscillation():
    "with adaptation and modest drive, the population cycles at 0.4 to 1.5 Hz and spends 30 to 70% of time up"
    s = stats["adaptation"]
    assert 0.4 < s[3] < 1.5 and 0.3 < s[0] < 0.7

test_noise_only()
test_oscillation()
print("All checks passed")

### Exercise: More drive, fewer down states

With b = 2, sweep the drive I over 0, 0.5, 1, 1.5 and 2 and record the fraction of time in the up state.

In [ ]:
upfrac = {}
for I in [0.0, 0.5, 1.0, 1.5, 2.0]:
    X, _ = ...
    upfrac[I] = (X > 0.5).mean()
print("fraction of time up by drive:", ", ".join(f"I = {I}: {v:.2f}" for I, v in upfrac.items()))

In [ ]:
import numpy as np
def test_sleep():
    "with no drive the population is almost always down"
    assert upfrac[0.0] < 0.1
def test_wake():
    "with drive 2 it is up more than 95% of the time"
    assert upfrac[2.0] > 0.95
def test_middle():
    "with drive 1 it alternates"
    assert 0.3 < upfrac[1.0] < 0.7

test_sleep()
test_wake()
test_middle()
print("All checks passed")

### Exercise: A peak below 1 Hz

Compute the power spectrum of population activity for I = 1 (sleep-like) and I = 2 (waking-like), each over 120 s, and compare the power below 2 Hz. Find the peak frequency of the sleep-like spectrum between 0.1 and 5 Hz.

In [ ]:
def spectrum(X, dt=0.002):
    x = X - X.mean()
    fr = np.fft.rfftfreq(len(x), dt); P = np.abs(np.fft.rfft(x)) ** 2 / len(x)
    return fr, P

fr, P_sleep = spectrum(run(I=1.0, b=2.0, T=120)[0])
_, P_wake = spectrum(run(I=2.0, b=2.0, T=120)[0])
m = (fr > 0.1) & (fr < 5)
peak = ...
low_sleep, low_wake = ...
print(f"sleep-like peak at {peak:.2f} Hz; power below 2 Hz: sleep {low_sleep:.3g}, waking {low_wake:.3g}")
plt.semilogy(fr[m], P_sleep[m], label="I = 1"); plt.semilogy(fr[m], P_wake[m], label="I = 2"); plt.xlabel("frequency (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_peak():
    "the sleep-like state has a spectral peak between 0.2 and 1.5 Hz"
    assert 0.2 < peak < 1.5
def test_power():
    "power below 2 Hz is at least 10 times greater in the sleep-like than the waking-like state"
    assert low_sleep > 10 * low_wake

test_peak()
test_power()
print("All checks passed")